## **MIN-HASHING AND LSH**

In [45]:
import re
import random
import time
import numpy as np
import pandas as pd
import math
from itertools import islice,combinations
from collections import Counter,defaultdict

In [46]:
def read_files(filepath:list[str]) -> dict[str,str]:
  filetext = {}
  for path in filepath:
    with open(path,"r",encoding="utf-8") as f:
      filetext[path] = f.read()
  return filetext

In [47]:
def tokenize(text:str) -> list[str]:
  return text.split()

def sentences(text: str) -> list[str]:
    return [s.strip() for s in text.split(" ") if s.strip()]

def sliding_window(sequence, k):
    it = iter(sequence)
    window = tuple(islice(it, k))
    if len(window) == k:
        yield window
    for element in it:
        window = window[1:] + (element,)
        yield window

In [48]:
def character_kgrams(text: str, k: int) -> list[tuple]:
    return set(sliding_window(text, k))

def word_kgrams(text: str, k: int) -> list[tuple]:
    return set(sliding_window(tokenize(text), k))

def sentence_kgrams(text: str, k: int) -> list[tuple]:
    return set(sliding_window(sentences(text), k))

## 1. **Create k-Grams**
You will construct several types of k-grams for all documents. All documents only have at
most 27 characters: all lowercase letters and space. The space counts as a character in
character k-grams.
Construct 2-grams based on characters for all documents.
Construct 3-grams based on characters for all documents.
Construct 2-grams based on words for all documents.
Remember that you should only store each k-gram once. Duplicates are ignored.

In [49]:
files = {"Doc1": "/content/drive/MyDrive/minhash/D1.txt", "Doc2":"/content/drive/MyDrive/minhash/D2.txt", "Doc3":"/content/drive/MyDrive/minhash/D3.txt", "Doc4":"/content/drive/MyDrive/minhash/D4.txt"}
raw = read_files(list(files.values()))
texts = {name: raw[path] for name, path in files.items()}

char_2grams = set()
for text in texts.values():
    char_2grams.update(character_kgrams(text, k=2))


char_3grams = set()
for text in texts.values():
    char_3grams.update(character_kgrams(text, k=3))


word_2grams = set()
for text in texts.values():
    word_2grams.update(word_kgrams(text, k=2))

print(f"Unique Char 2-grams")
print(char_2grams)

print(f"Unique Char 3-grams")
print(char_3grams)

print(f"Unique Word 2-grams")
print(word_2grams)

Unique Char 2-grams
{('b', 'r'), (' ', 't'), ('i', 'd'), ('i', 'q'), ('u', 'g'), ('c', 'l'), ('a', 'z'), ('e', 'c'), ('f', 'o'), ('r', 't'), ('c', 'y'), ('a', 'u'), ('r', 'm'), ('v', 'i'), ('t', 'r'), ('i', 'l'), ('i', 'c'), ('a', 'i'), ('p', 'l'), ('o', 'i'), ('k', 'n'), ('r', ' '), ('g', 'o'), ('i', 'g'), ('u', 'm'), ('n', 'i'), ('c', 't'), ('b', 'u'), ('u', ' '), (' ', 'x'), ('r', 'e'), ('k', 'i'), ('d', 'i'), ('b', 'i'), ('k', 'l'), ('r', 'a'), ('a', 'p'), ('o', 'p'), ('i', 't'), ('o', 'f'), ('c', ' '), ('h', 'o'), ('b', 'y'), ('k', 'y'), ('p', 't'), ('w', 'o'), ('u', 'e'), ('a', 'f'), ('t', 'z'), ('i', 'm'), ('t', 'u'), ('y', 'i'), ('u', 'a'), ('g', 'n'), ('t', 'i'), ('i', ' '), ('c', 'e'), ('p', ' '), ('j', 'o'), ('c', 'a'), ('f', 'y'), ('g', 'h'), ('t', 'y'), ('l', 's'), (' ', 'w'), ('x', ' '), ('v', 'e'), ('i', 'e'), ('e', 'x'), ('p', 'e'), (' ', 'k'), ('w', 's'), ('g', 'l'), ('o', 'b'), ('p', 'a'), ('k', ' '), ('g', 'y'), ('g', 'g'), ('t', 'f'), ('r', 'r'), ('f', 't'), ('x', '

## 1.A Using 3-grams to build a min-hash signature for documents D1 and D2 using t = 20, 60, 150, 300, 600 hash functions. For each value of ( t ), report the approximate Jaccard similarity between the pair of documents D1 and D2, estimating the Jaccard similarity

In [50]:
def jaccard_similarity(set_a: set, set_b: set) -> float:
    intersection = len(set_a & set_b)
    union = len(set_a | set_b)
    return intersection / union if union != 0 else 0.0

def make_hash_fn(a: int, b: int, p: int, m: int):
    """Hash function of the form h(x) = (a*x + b) mod p mod m"""
    return lambda x: ((a * x + b) % p) % m

def minhash_signature(kgrams: set, hash_fns: list) -> list[int]:
    sig = [float('inf')] * len(hash_fns)
    for gram in kgrams:
        hashed = hash(gram) & 0x7FFFFFFF  # ensure positive int
        for i, fn in enumerate(hash_fns):
            val = fn(hashed)
            if val < sig[i]:
                sig[i] = val
    return sig

def approx_jaccard_minhash(sig_a: list, sig_b: list) -> float:
    t = len(sig_a)
    matches = sum(1 for i in range(t) if sig_a[i] == sig_b[i])
    return matches / t

In [51]:
kgram_sets = {
    "Char 2-grams": {name: character_kgrams(text, 2) for name, text in texts.items()},
    "Char 3-grams": {name: character_kgrams(text, 3) for name, text in texts.items()},
    "Word 2-grams": {name: word_kgrams(text, 2)      for name, text in texts.items()},
}

In [52]:
t_values = [20, 60, 150, 300, 600]

# Pre-generate enough hash functions for max t
max_t_value = max(t_values)
PRIME = (1 << 31) - 1
M = 1 << 20

random.seed(42)
A = [random.randint(1, PRIME - 1) for _ in range(max_t_value)]
B = [random.randint(0, PRIME - 1) for _ in range(max_t_value)]

doc1_3grams = kgram_sets["Char 3-grams"]["Doc1"]
doc2_3grams = kgram_sets["Char 3-grams"]["Doc2"]

true_jaccard = jaccard_similarity(doc1_3grams, doc2_3grams)
print(f"\nTrue Jaccard (D1 vs D2, Char 3-grams): {true_jaccard:.4f}\n")

print(f"{'t':<8} {'Approx Jaccard Similarity':>15}")
print("-" * 25)
for t in t_values:
    hash_fns = [make_hash_fn(A[i], B[i], PRIME, M) for i in range(t)]
    sig_d1 = minhash_signature(doc1_3grams, hash_fns)
    sig_d2 = minhash_signature(doc2_3grams, hash_fns)
    approx = approx_jaccard_minhash(sig_d1, sig_d2)
    print(f"{t:<8} {approx:>15.4f}")


True Jaccard (D1 vs D2, Char 3-grams): 0.9780

t        Approx Jaccard Similarity
-------------------------
20                1.0000
60                1.0000
150               1.0000
300               0.9867
600               0.9867


## 1.B Compute the Jaccard similarity between all pairs of documents for each type of k-gram. You should report: 3 × 6 = 18 different numbers.

In [53]:
print(f"{'K-gram Type':<15} {'Document Pair':<15} {'Jaccard Similarity':>18}")
print("-" * 50)

for kgram_type, doc_sets in kgram_sets.items():
    doc_names = list(doc_sets.keys())
    for doc_a, doc_b in combinations(doc_names, 2):
        sim = jaccard_similarity(doc_sets[doc_a], doc_sets[doc_b])
        print(f"{kgram_type:<15} {doc_a} vs {doc_b:<5} {sim:>18.4f}")
    print()

K-gram Type     Document Pair   Jaccard Similarity
--------------------------------------------------
Char 2-grams    Doc1 vs Doc2              0.9811
Char 2-grams    Doc1 vs Doc3              0.8157
Char 2-grams    Doc1 vs Doc4              0.6444
Char 2-grams    Doc2 vs Doc3              0.8000
Char 2-grams    Doc2 vs Doc4              0.6413
Char 2-grams    Doc3 vs Doc4              0.6530

Char 3-grams    Doc1 vs Doc2              0.9780
Char 3-grams    Doc1 vs Doc3              0.5804
Char 3-grams    Doc1 vs Doc4              0.3051
Char 3-grams    Doc2 vs Doc3              0.5680
Char 3-grams    Doc2 vs Doc4              0.3059
Char 3-grams    Doc3 vs Doc4              0.3121

Word 2-grams    Doc1 vs Doc2              0.9408
Word 2-grams    Doc1 vs Doc3              0.1823
Word 2-grams    Doc1 vs Doc4              0.0302
Word 2-grams    Doc2 vs Doc3              0.1737
Word 2-grams    Doc2 vs Doc4              0.0303
Word 2-grams    Doc3 vs Doc4              0.0161



##2. **Min-Hashing**:
We will consider a hash family ( H ) so that any hash function ( h ∈H ) maps from:
h : k-grams → [m]
for ( m ) large enough (To be extra cautious, use ( m greater than 10,000 )).


## 2.A Using 3-grams to build a min-hash signature for documents D1 and D2 using: t = 20, 60, 150, 300, 600 hash functions. For each value of ( t ), report the approximate Jaccard similarity between the pair of documents D1 and D2, estimating the Jaccard similarity. You should report 5 numbers.

In [54]:
t_values = [20, 60, 150, 300, 600]

# Pre-generate enough hash functions for max t
max_t_value = max(t_values)
PRIME = (1 << 31) - 1
M = 1 << 20

random.seed(42)
A = [random.randint(1, PRIME - 1) for _ in range(max_t_value)]
B = [random.randint(0, PRIME - 1) for _ in range(max_t_value)]

doc1_3grams = kgram_sets["Char 3-grams"]["Doc1"]
doc2_3grams = kgram_sets["Char 3-grams"]["Doc2"]

print(f"Value of m is {M}")
print("-" * 35)

print(f"{'t':<8} {'Approx Jaccard Similarity':>15}")
print("-" * 35)
for t in t_values:
    hash_fns = [make_hash_fn(A[i], B[i], PRIME, M) for i in range(t)]
    sig_d1 = minhash_signature(doc1_3grams, hash_fns)
    sig_d2 = minhash_signature(doc2_3grams, hash_fns)
    approx = approx_jaccard_minhash(sig_d1, sig_d2)
    print(f"{t:<8} {approx:>15.4f}")

Value of m is 1048576
-----------------------------------
t        Approx Jaccard Similarity
-----------------------------------
20                1.0000
60                1.0000
150               1.0000
300               0.9867
600               0.9867


## 2.B What seems to be a good value for ( t )? You may run more experiments. Justify your answer in terms of both accuracy and time.

In [55]:
diff_t_values = [10, 20, 40, 60, 100, 150, 200, 300, 450, 600, 800, 1000]

true_jaccard = jaccard_similarity(doc1_3grams, doc2_3grams)

max_t = max(diff_t_values)
random.seed(42)
A_ext = [random.randint(1, PRIME - 1) for _ in range(max_t)]
B_ext = [random.randint(0, PRIME - 1) for _ in range(max_t)]

print(f"True Jaccard (Doc1 vs Doc2, Char 3-grams): {true_jaccard:.4f}\n")
print(f"{'t':<8} {'Approx Jaccard Sim':>15} {'Error':>10} {'Time (ms)':>12}")
print("-" * 55)

errors = []
times  = []

for t in diff_t_values:
    hash_fns = [make_hash_fn(A_ext[i], B_ext[i], PRIME, M) for i in range(t)]

    starttime = time.perf_counter()
    sig_d1 = minhash_signature(doc1_3grams, hash_fns)
    sig_d2 = minhash_signature(doc2_3grams, hash_fns)
    approx = approx_jaccard_minhash(sig_d1, sig_d2)
    elapsed_time = (time.perf_counter() - starttime) * 1000

    error = abs(approx - true_jaccard)
    errors.append(error)
    times.append(elapsed_time)

    print(f"{t:<8} {approx:>15.4f} {error:>10.4f} {elapsed_time:>12.2f}")

True Jaccard (Doc1 vs Doc2, Char 3-grams): 0.9780

t        Approx Jaccard Sim      Error    Time (ms)
-------------------------------------------------------
10                1.0000     0.0220         6.88
20                1.0000     0.0220        12.33
40                1.0000     0.0220        29.48
60                1.0000     0.0220        37.97
100               1.0000     0.0220        60.37
150               0.9933     0.0154        95.30
200               0.9850     0.0070       138.44
300               0.9767     0.0013       221.40
450               0.9756     0.0024       304.35
600               0.9767     0.0013       394.52
800               0.9750     0.0030       571.18
1000              0.9750     0.0030       677.14


# Observation on different values of t:
In terms of accuracy, it is evident that when t value is set to 100, the error has reduced from 0.0113 to 0.0020. When t value is increased to 150, the error remains the same and then it starts fluctuating again in no particular order. So, adding more hash functions beyond 100 doesn’t give any definite improvement in terms of accuracy.
In terms of time, it is seen that t=100 (error 0.0020 takes around 61.97ms whereas t=450 where the error is the least(0.0002) takes around 276.35. Though it takes more than 4 times the time, the error has only reduced by 0.0018 which is not very significant.
Therefore, after t=100, though the time taken grows linearly, the accuracy doesn’t show significant improvement.


## 3. **LSH**:
Consider computing an LSH using: t = 160
hash functions. We want to find all document pairs with Jaccard similarity above: τ = .7

In [56]:
t   = 160
sim_threshold = 0.7

# Step 1: Generate 160 actual hash functions
random.seed(42)
A160 = [random.randint(1, PRIME - 1) for _ in range(t)]
B160 = [random.randint(0, PRIME - 1) for _ in range(t)]
hash_fns_160 = [make_hash_fn(A160[i], B160[i], PRIME, M) for i in range(t)]


# Step 2: Build MinHash signatures for all 4 docs
signatures = {}
doc_3grams = kgram_sets["Char 3-grams"]
doc_names  = list(doc_3grams.keys())
candidates = []
for r_try in range(1, t + 1):
    if t % r_try == 0:
        b_try  = t // r_try
        thresh = (1 / b_try) ** (1 / r_try)
        candidates.append((r_try, b_try, thresh, abs(thresh - sim_threshold)))

candidates.sort(key=lambda x: x[3])
r, b, approx_thresh, _ = candidates[0]

for doc_name, grams in doc_3grams.items():
    signatures[doc_name] = minhash_signature(grams, hash_fns_160)

print(f"\nSignature matrix  ({t} hash functions × {len(signatures)} docs):")
print(f"{'Doc':<8}", end="")
for i in range(min(6, t)):
    print(f"  h{i+1:>3}", end="")
print("  ...")
for doc, sig in signatures.items():
    print(f"{doc:<8}", end="")
    for v in sig[:6]:
        print(f"  {v:>4}", end="")
    print("  ...")

# Step 3: Apply LSH banding
def lsh_band(signatures, b, r):
    candidate_pairs = set()
    for band_idx in range(b):                        # for each band
        buckets = {}
        start = band_idx * r
        end   = start + r
        for doc in signatures:
            band_key = tuple(signatures[doc][start:end])  # r hash values
            buckets.setdefault(band_key, []).append(doc)
        for bucket_docs in buckets.values():
            if len(bucket_docs) > 1:                 # collision = candidate pair
                for i in range(len(bucket_docs)):
                    for j in range(i+1, len(bucket_docs)):
                        candidate_pairs.add(tuple(sorted([bucket_docs[i], bucket_docs[j]])))
    return candidate_pairs

candidate_pairs = lsh_band(signatures, b, r)

# Step 4: Report results
print(f"\n {'Pair':<20} {'True Jaccard':>14} {'Candidate?':>12}")
print("-" * 48)
for doc_a, doc_b in combinations(doc_names, 2):
    s       = jaccard_similarity(doc_3grams[doc_a], doc_3grams[doc_b])
    is_cand = tuple(sorted([doc_a, doc_b])) in candidate_pairs
    print(f"{doc_a+' vs '+doc_b:<20} {s:>14.4f} {'Yes' if is_cand else 'No':>12}")


Signature matrix  (160 hash functions × 4 docs):
Doc       h  1  h  2  h  3  h  4  h  5  h  6  ...
Doc1       275  1702  3324  5588   568  1208  ...
Doc2       275  1702  3324  5588   568  1208  ...
Doc3       275  1702  3324   314   568  1208  ...
Doc4       275  1525  4609  2592  2024  1737  ...

 Pair                   True Jaccard   Candidate?
------------------------------------------------
Doc1 vs Doc2                 0.9780          Yes
Doc1 vs Doc3                 0.5804           No
Doc1 vs Doc4                 0.3051           No
Doc2 vs Doc3                 0.5680           No
Doc2 vs Doc4                 0.3059           No
Doc3 vs Doc4                 0.3121           No


## 3.A
Use the formula mentioned in class and the notes to estimate the best values of hash
functions ( b ) within each of the ( r ) bands to provide the S-curve:
f(s) = 1 − (1 − sb)r
with good separation at (τ ). Report these values.

In [57]:
t   = 160
sim_threshold = 0.7

# Analysing the top 8 valid (r,b) pairs sorted by closeness to sim_threshold to identify the best values for r and b
print(f"\nTop 8 valid (r, b) pairs sorted by |threshold - tau|:")
print(f"\n{'r':>6} {'b':>6} {'r*b':>6} {'(1/b)^(1/r)':>14} {'|diff|':>10}")
print("-" * 46)
candidates = []
for r_try in range(1, t + 1):
    if t % r_try == 0:
        b_try  = t // r_try
        thresh = (1 / b_try) ** (1 / r_try)
        candidates.append((r_try, b_try, thresh, abs(thresh - sim_threshold)))

candidates.sort(key=lambda x: x[3])

for r_try, b_try, thresh, diff in candidates[:8]:
    print(f"{r_try:>6} {b_try:>6} {r_try*b_try:>6} {thresh:>14.4f} {diff:>10.4f}")

r, b, approx_thresh, _ = candidates[0]

print(f"\nBest value of r (rows/band)    : {r}")
print(f"\nBest value of b (bands)        : {b}")
print(f"\nS-curve threshold with good separation at tau: (1/b)^(1/r) = {approx_thresh:.4f}  (tau = {sim_threshold})")


Top 8 valid (r, b) pairs sorted by |threshold - tau|:

     r      b    r*b    (1/b)^(1/r)     |diff|
----------------------------------------------
     8     20    160         0.6877     0.0123
    10     16    160         0.7579     0.0579
    16     10    160         0.8660     0.1660
     5     32    160         0.5000     0.2000
    20      8    160         0.9013     0.2013
    32      5    160         0.9509     0.2509
    40      4    160         0.9659     0.2659
    80      2    160         0.9914     0.2914

Best value of r (rows/band)    : 8

Best value of b (bands)        : 20

S-curve threshold with good separation at tau: (1/b)^(1/r) = 0.6877  (tau = 0.7)


## 3.B
Using your choice of ( r ) and ( b ) and ( f(⋅) ), what is the probability of each pair of the four
documents (using 3-grams) being estimated to have a similarity greater than ( τ )? Report 6
numbers.

In [58]:
def f_scurve(s, r, b):
    return 1 - (1 - s**r)**b

print(f"Using b={b} (bands) & r={r} (rows/band)")
print(f"\nf(s) = 1 - (1 - s^{r})^{b}\n")

doc_3grams = kgram_sets["Char 3-grams"]
doc_names  = list(doc_3grams.keys())

print(f"{'Pair':<20} {'f(s)':>12} {'Similarity>tau?':>20}")
print("-" * 60)

for doc_a, doc_b in combinations(doc_names, 2):
    s    = jaccard_similarity(doc_3grams[doc_a], doc_3grams[doc_b])
    prob = f_scurve(s, r, b)
    flag = "Yes" if s >= sim_threshold else "No"
    print(f"{doc_a+' vs '+doc_b:<20} {prob:>12.6f} {flag:>10}")

Using b=20 (bands) & r=8 (rows/band)

f(s) = 1 - (1 - s^8)^20

Pair                         f(s)      Similarity>tau?
------------------------------------------------------------
Doc1 vs Doc2             1.000000        Yes
Doc1 vs Doc3             0.228224         No
Doc1 vs Doc4             0.001500         No
Doc2 vs Doc3             0.195880         No
Doc2 vs Doc4             0.001532         No
Doc3 vs Doc4             0.001800         No


## 4. **Min-Hashing on MovieLens dataset**
Implement Min-Hashing on the older MovieLens 100k dataset (5MB), which consists of a set
of 943 users who have rated 1682 movies. You can download the data from:
http://www.grouplens.org/node/73
Read the Readme file for details about the data, and process it as you need.
For this exercise, we only care about the set of movies that a user has rated and not the
ratings. We want to compute the Jaccard similarity between the users.
Compute the exact Jaccard similarity for all pairs of users, and output the pairs of users that
have a similarity of at least 0.5.
Then, compute the min-hash signatures for the users and compute the approximate Jaccard
similarity.
Use:

50 hash functions
100 hash functions
200 hash functions

For each value, output the pairs that have an estimated similarity of at least 0.5 and report
the number of false positives and false negatives that you obtain.
For the false positives and negatives, report the averages for 5 different runs.

In [59]:
#Loading older MovieLens 100k dataset
df = pd.read_csv('/content/drive/MyDrive/u.data', sep='\t',
                 names=['user_id', 'movie_id', 'rating', 'timestamp'])

print(f"MovieLens 100k Dataset loaded with")
print(f"  Users  : {df['user_id'].nunique()}")
print(f"  Movies : {df['movie_id'].nunique()}")
print(f"  Ratings: {len(df)}")

# Building set of users and the movies rated by them
user_movies = defaultdict(set)
for _, row in df.iterrows():
    user_movies[int(row['user_id'])].add(int(row['movie_id']))

users      = sorted(user_movies.keys())
all_movies = sorted(df['movie_id'].unique())
n_users    = len(users)
n_movies   = len(all_movies)

print("\nComputing exact Jaccard similarities...")
exact_similar = set()   # pairs with similarity >= 0.5
exact_jaccard = {}      # all pairs

for i in range(n_users):
    for j in range(i + 1, n_users):
        u1, u2 = int(users[i]), int(users[j])
        sim = jaccard_similarity(user_movies[u1], user_movies[u2])
        exact_jaccard[(u1, u2)] = sim
        if sim >= 0.5:
            exact_similar.add((u1, u2))

print(f"\nNumber of exact similar pairs (Jaccard Similarity >= 0.5): {len(exact_similar)}")
print(f"\nPairs of users that have exact similarity of atleast 0.5: {exact_similar}")

PRIME = (1 << 31) - 1
M     = n_movies + 1

# Computing min-hash signatures
t_values  = [50,100,200]
threshold = 0.5
n_runs    = 5

print("\nComputing min-hash signatures....")


results_summary = {}

for t in t_values:
    print(f"\n t = {t} hash functions:")
    print(f"--------------------------")

    fp_list, fn_list = [], []

    for run in range(n_runs):
        random.seed(run * 100 + t)   # different seed each run
        A = [random.randint(1, PRIME - 1) for _ in range(t)]
        B = [random.randint(0, PRIME - 1) for _ in range(t)]
        hash_fns = [make_hash_fn(A[i], B[i], PRIME, M) for i in range(t)]

        # Building signatures
        signatures = {}
        for u in users:
            signatures[u] = minhash_signature(user_movies[u], hash_fns)

        # Finding approximate similar pairs
        approx_similar = set()
        for i in range(n_users):
            for j in range(i + 1, n_users):
                u1, u2 = int(users[i]), int(users[j])
                sim = approx_jaccard_minhash(signatures[u1], signatures[u2])
                if sim >= threshold:
                    approx_similar.add((u1, u2))

        # False positives and false negatives
        fp = len(approx_similar - exact_similar)    # counted but not similar
        fn = len(exact_similar - approx_similar)    # similar but missed
        fp_list.append(fp)
        fn_list.append(fn)

        print(f"\nRun {run+1}: Estimated similar pairs={len(approx_similar):>4}, "
              f"False Positives={fp:>3}, False Negatives={fn:>3}")

    avg_fp = sum(fp_list) / n_runs
    avg_fn = sum(fn_list) / n_runs

    print(f"\nPairs of users that have estimated similarity of atleast 0.5: {approx_similar}")
    print(f"\nAverage False Positives over {n_runs} runs: {avg_fp:.2f}")
    print(f"\nAverage False Negatives over {n_runs} runs: {avg_fn:.2f}")



MovieLens 100k Dataset loaded with
  Users  : 943
  Movies : 1682
  Ratings: 100000

Computing exact Jaccard similarities...

Number of exact similar pairs (Jaccard Similarity >= 0.5): 10

Pairs of users that have exact similarity of atleast 0.5: {(451, 489), (328, 788), (600, 826), (554, 764), (674, 879), (800, 879), (197, 826), (489, 587), (408, 898), (197, 600)}

Computing min-hash signatures....

 t = 50 hash functions:
--------------------------

Run 1: Estimated similar pairs= 183, False Positives=173, False Negatives=  0

Run 2: Estimated similar pairs= 853, False Positives=845, False Negatives=  2

Run 3: Estimated similar pairs= 353, False Positives=347, False Negatives=  4

Run 4: Estimated similar pairs= 304, False Positives=295, False Negatives=  1

Run 5: Estimated similar pairs= 147, False Positives=140, False Negatives=  3

Pairs of users that have estimated similarity of atleast 0.5: {(258, 856), (59, 537), (551, 627), (273, 673), (407, 643), (301, 363), (151, 537), (53

## 5. **LSH on MovieLens dataset**
Break up the signature table into ( b ) bands with ( r ) hash functions per band and
implement Locality Sensitive Hashing.
The goal is to find candidate pairs with a similarity of at least 0.6.
Experiment with:
Report the number of false positives and false negatives, taking the average over 5 runs.
How do these numbers change if we want a similarity of at least 0.8?

In [60]:
experiments = [
    (50,  5, 10),   # t=50,  r=5, b=10
    (100, 5, 20),   # t=100, r=5, b=20
    (200, 5, 40),   # t=200, r=5, b=40
    (200, 10, 20),  # t=200, r=10, b=20
]

n_runs    = 5
M         = n_movies + 1

# ── LSH BANDING ───────────────────────────────────────────────────────────────
def lsh_band(signatures, b, r):
    candidate_pairs = set()
    user_list = list(signatures.keys())
    for band_idx in range(b):
        buckets = {}
        start = band_idx * r
        end   = start + r
        for user in user_list:
            band_key = tuple(signatures[user][start:end])
            buckets.setdefault(band_key, []).append(user)
        for bucket_users in buckets.values():
            if len(bucket_users) > 1:
                for i in range(len(bucket_users)):
                    for j in range(i + 1, len(bucket_users)):
                            pair = tuple(sorted([bucket_users[i], bucket_users[j]]))
                            candidate_pairs.add(pair)
    return candidate_pairs

# ── COMPUTE EXACT SIMILAR PAIRS FOR BOTH THRESHOLDS ──────────────────────────
print("Computing exact Jaccard similarities...")
exact_similar_06 = set()   # pairs with J >= 0.6
exact_similar_08 = set()   # pairs with J >= 0.8

for i in range(n_users):
    for j in range(i + 1, n_users):
        u1, u2 = int(users[i]), int(users[j])
        sim = jaccard_similarity(user_movies[u1], user_movies[u2])
        exact_jaccard[(u1, u2)] = sim
        if sim >= 0.6:
            exact_similar_06.add((u1, u2))
        if sim >= 0.8:
            exact_similar_08.add((u1,u2))

print(f"Exact similar pairs with Jac Sim >= 0.6: {len(exact_similar_06)}")
print(f"Exact similar pairs with Jac Sim >= 0.8: {len(exact_similar_08)}")


def run_experiment(t, r, b, exact_similar, tau, n_runs=5):
    fp_list, fn_list, verified_fp_list,verified_fn_list,candidate_list = [], [], [],[],[]

    for run in range(n_runs):
        random.seed(run * 100 + t)
        A        = [random.randint(1, PRIME - 1) for _ in range(t)]
        B        = [random.randint(0, PRIME - 1) for _ in range(t)]
        hash_fns = [make_hash_fn(A[i], B[i], PRIME, M) for i in range(t)]

        # Build signatures
        signatures = {}
        for u in users:
            signatures[u] = minhash_signature(user_movies[u], hash_fns)

        # LSH candidate pairs
        candidate_pairs = lsh_band(signatures, b, r)

        falsepositive = len(candidate_pairs - exact_similar)
        falsenegative = len(exact_similar - candidate_pairs)
        fp_list.append(falsepositive)
        fn_list.append(falsenegative)
        candidate_list.append(len(candidate_pairs))

        #Verification of candidate pairs by comparing the true jaccard value against the similarity threshold
        verified_pairs = set()
        for (u1, u2) in candidate_pairs:
            true_sim = jaccard_similarity(user_movies[u1], user_movies[u2])
            if true_sim >= tau:
                verified_pairs.add((u1, u2))

        fp = len(verified_pairs - exact_similar)
        fn = len(exact_similar - verified_pairs)
        verified_fp_list.append(fp)
        verified_fn_list.append(fn)


        print(f" Run {run+1}: Candidate pairs={len(candidate_pairs):>5}, "
              f" False Positive={falsepositive:>4}, False Negative={falsenegative:>4},"
              f" Verified pairs={len(verified_pairs):>5}, False Positive={fp:>4}, False Negative={fn:>4}")



    return sum(fp_list)/n_runs, sum(fn_list)/n_runs,sum(verified_fp_list)/n_runs,sum(verified_fn_list)/n_runs

print(f"\n{'='*65}")
print(f"RESULTS FOR tau = 0.6")
print(f"{'='*65}")


for t, r, b in experiments:
    print(f"\nt: {t:<8} r: {r:<6} b: {b:<6}")
    avg_fp, avg_fn, avg_fp_v,avg_fn_v = run_experiment(t, r, b, exact_similar_06, 0.6)
    print("\nCandidate Pairs:")
    print(f"Average False Positives: {avg_fp:>10.2f} Average False Negatives: {avg_fn:>10.2f}")
    print("\nVerified Pairs:")
    print(f"Average False Positives: {avg_fp_v:>10.2f} Average False Negatives: {avg_fn_v:>10.2f}")

print(f"\n{'='*65}")
print(f"RESULTS FOR tau = 0.8")
print(f"{'='*65}")


for t, r, b in experiments:
    print(f"\nt: {t:<8} r: {r:<6} b: {b:<6}")
    avg_fp, avg_fn, avg_fp_v,avg_fn_v = run_experiment(t, r, b, exact_similar_08, 0.8)
    print("\nCandidate Pairs:")
    print(f"Average False Positives: {avg_fp:>10.2f} Average False Negatives: {avg_fn:>10.2f}")
    print("\nVerified Pairs:")
    print(f"Average False Positives: {avg_fp_v:>10.2f} Average False Negatives: {avg_fn_v:>10.2f}")


Computing exact Jaccard similarities...
Exact similar pairs with Jac Sim >= 0.6: 3
Exact similar pairs with Jac Sim >= 0.8: 1

RESULTS FOR tau = 0.6

t: 50       r: 5      b: 10    
 Run 1: Candidate pairs=  541,  False Positive= 539, False Negative=   1, Verified pairs=    2, False Positive=   0, False Negative=   1
 Run 2: Candidate pairs= 1731,  False Positive=1728, False Negative=   0, Verified pairs=    3, False Positive=   0, False Negative=   0
 Run 3: Candidate pairs=  684,  False Positive= 682, False Negative=   1, Verified pairs=    2, False Positive=   0, False Negative=   1
 Run 4: Candidate pairs=  736,  False Positive= 734, False Negative=   1, Verified pairs=    2, False Positive=   0, False Negative=   1
 Run 5: Candidate pairs=  721,  False Positive= 718, False Negative=   0, Verified pairs=    3, False Positive=   0, False Negative=   0

Candidate Pairs:
Average False Positives:     880.20 Average False Negatives:       0.60

Verified Pairs:
Average False Positives:  

# Observation on how these numbers change if we want a similarity of at least 0.8?
•	The count of similar pairs whose Jaccard Similarity exceeds the threshold has reduced from 3 to 1. This implies that with a higher threshold, the number of true similar pairs reduces.

•	False Negatives have decreased with increasing threshold. This is because highly similar pairs (Threshold=0.8) is easier for LSH to find as they collide in bands more reliably.
